# DevAPI02.ipynb: Exception handling for `estatpy.api.get_csv_data()`

Because estatpy.api.get_csv_data() will use internet requests and other operations that will likely through exceptions, exception handling must be added before release.

## Load preamble matter

In [1]:
%%capture capt
# %%capture puts the output in 'capt' from which it can be ignored or printed
%pip install pandas
%pip install python-dotenv # for using local environment variables
%pip install fsspec # to help pandas IO. See https://pandas.pydata.org/docs/user_guide/io.html#reading-writing-remote-files
%pip install requests

## Technical references for this task

* [Python tutorial on errors and exceptions](https://docs.python.org/3.14/tutorial/errors.html)
    * [8.10. Enriching Exceptions with Notes](https://docs.python.org/3.14/tutorial/errors.html#enriching-exceptions-with-notes)
* [Python built-in exceptions](https://docs.python.org/3.14/library/exceptions.html)
* [Requests: HTTP for Humans](https://requests.readthedocs.io/en/latest/)
    * [Response Status Codes](https://requests.readthedocs.io/en/latest/user/quickstart/#response-status-codes)
    * [Developer Interface / Exceptions](https://requests.readthedocs.io/en/latest/api/#exceptions)
* [e-stat-api/adaptor on GitHub](https://github.com/e-stat-api/adaptor)

## THE FUNCTION

In [5]:
import pandas as pd
import os
from dotenv import load_dotenv
import requests
import tempfile
import re
import datetime
def get_csv_data(url, description = datetime.datetime.now()):
    try:
        load_dotenv()
    except (FileNotFoundError,IOError) as e:
        e.add_note('Environment variable file (.env) not found. See README.')
        raise
    
    try:
        app_id = os.environ['ESTAT_APP_ID']
    except KeyError as e:
        e.add_note('Environment variable ESTAT_APP_ID not found. See README.')
        raise

    if app_id == None:
        raise OSError("Value of environment variable 'ESTAT_APP_ID' not found. See README.")
    
    url_split = url.split("appId=")
    if len(url_split) != 2:
        raise Exception("Invalid API url")
    url = url_split[0] + "appId=" + app_id + url_split[1]

    # the csv has several rows of metadata terminated by a row starting with "VALUE".
    # The data table starts on the next row.
    # Put the metadata in a StringIO
    result = {}
    try:
        with requests.get(url,stream=False) as estatresponse: # chunking in iter_lines doesn't work for stream=True
            estatresponse.raise_for_status()

            if estatresponse.encoding is None:
                estatresponse.encoding = 'utf-8'
            estatlines = estatresponse.iter_lines(chunk_size=1024, decode_unicode=True)
            with tempfile.NamedTemporaryFile(mode='w',delete_on_close=False,encoding = 'utf-8') as fheader:
                with tempfile.NamedTemporaryFile(mode='w',delete_on_close=False,encoding = 'utf-8') as fp:
                    inheader = True
                    colnum = 0
                    for line in estatlines:
                        if inheader == True:
                            #count columns
                            fields = re.split('","',line)
                            if len(fields) > colnum :
                                colnum = len(fields)
                            fheader.write(line)
                            fheader.write("\n")
                            if( line.startswith('"VALUE"')):
                                inheader = False
                                fheader.flush()
                                fheader.seek(0)
                        else:
                            fp.write(line)
                            fp.write("\n")
                    fheader.close()
                    fp.close()
                    if inheader == True:
                        errmsg = "The stream that e-Stat returned lacks a 'VALUE' line. See temp file: " + fheader.name
                        raise Exception(errmsg)
                    dfHeader = pd.read_csv(fheader.name, names = range(colnum))
                    dfHeader = dfHeader.dropna(axis=1, how = "all")
                    dfMain = pd.read_csv(fp.name)
                    result['Description'] = description
                    result['Header'] = dfHeader
                    result['Main'] = dfMain

    except requests.RequestException as e:
            raise
    
    return result

# Labor force url:
enurl = 'http://api.e-stat.go.jp/rest/3.0/app/getSimpleStatsData?appId=&lang=E&statsDataId=0003005798&metaGetFlg=Y&cntGetFlg=N&explanationGetFlg=Y&annotationGetFlg=Y&sectionHeaderFlg=1&replaceSpChars=0'

# bad url
badurl1 = 'http://api.e-stat.go.jp/rest/3.0/app/getSimpleStatsData?lang=E&statsDataId=0003005798&metaGetFlg=Y&cntGetFlg=N&explanationGetFlg=Y&annotationGetFlg=Y&sectionHeaderFlg=1&replaceSpChars=0'


try:        
    dfs = get_csv_data(enurl,description=datetime.datetime.now())
    print(dfs.get('Description'))
    print(dfs.get('Header'))
except Exception as e:
    print(type(e))
    print(e.args)
    print(e.with_traceback)




<class 'Exception'>
("The stream that e-Stat returned lacks a 'VALUE' line. See temp file: C:\\Users\\Alan\\AppData\\Local\\Temp\\tmpnn9d5g48",)
<built-in method with_traceback of Exception object at 0x00000240FFA392A0>


## Tests of exception handling

### Move `.env` file to `.mynotes` folder to disrupt `load_dotenv()`

VS Code had to be closed and restarted to throw an error. `load_dotenv()` was not disrupted but `app_id = os.environ['ESTAT_APP_ID']` threw an exception with the following message:

```
KeyError: 'ESTAT_APP_ID'
Environment variable ESTAT_APP_ID not found. See README.
```

Returning `.env` file to project root fixed this error.

### Alter `ESTAT_APP_ID` to `ESTAT_APP_ID_T`

After restarting VS Code, `app_id = os.environ['ESTAT_APP_ID']` threw an exception with the same message.

### Run with `badurl1` (No `appId=`)

Result:

```
<class 'Exception'>
('Invalid API url',)
<built-in method with_traceback of Exception object at 0x00000202BEF67340>
```

### Bad appId (run after restarting VS Code to clear environment variable)

Result:

`requests.get` returned a valid response with the following message (recovered from the temp file):

```
"RESULT"
"STATUS","100"
"ERROR_MSG","Authentication failed. Please check that your appID are correct."
"DATE","2026-03-05T10:15:55.716+09:00"
```

`dfMain = pd.read_csv(fp.name)` raised the following error because the stream pointer was at the end of the stream:

```
<class 'pandas.errors.EmptyDataError'>
('No columns to parse from file',)
<built-in method with_traceback of EmptyDataError object at 0x000002409BBFE740>
```

This should be caught before calling `pandas.read_csv()`.

Remedy: Check for 'inheader == True` after `fp.close()`. Raise exception if True. Result for invalid appID:

```
<class 'Exception'>
("The stream that e-Stat returned lacks a 'VALUE' line. See temp file: (omitted)",)
<built-in method with_traceback of Exception object at 0x00000240FFA392A0>
```